# OCD data via ArcGIS

importing the libraries

In [ ]:
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium

In [ ]:
SERVICE_URL = (
    "https://gis.emnrd.nm.gov/arcgis/rest/services/OCDView/Wells_Public/FeatureServer/0/query" # worked
)

def fetch_ocd(where="1=1", offset=0, batch=2000):
    params = {
        "where": where,
        "outFields": "*",
        "f": "json",
        "resultOffset": offset,
        "resultRecordCount": batch,
        "geometryType": "esriGeometryPoint",
        "spatialRel": "esriSpatialRelIntersects",
        "outSR": "4326",
        "returnGeometry": "true",
    }
    r = requests.get(SERVICE_URL, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

# Paginate through all records
all_features = []
offset = 0
while True:
    data = fetch_ocd(offset=offset)
    features = data.get("features", [])
    if not features:
        break
    all_features.extend(features)
    offset += len(features)
    print(f"  fetched {len(all_features)} records so far...")
    if not data.get("exceededTransferLimit", False):
        break

print(f"Total records: {len(all_features)}")


  fetched 2000 records so far...
  fetched 4000 records so far...
  fetched 6000 records so far...
  fetched 8000 records so far...
  fetched 10000 records so far...
  fetched 12000 records so far...
  fetched 14000 records so far...
  fetched 16000 records so far...
  fetched 18000 records so far...
  fetched 20000 records so far...
  fetched 22000 records so far...
  fetched 24000 records so far...
  fetched 26000 records so far...
  fetched 28000 records so far...
  fetched 30000 records so far...
  fetched 32000 records so far...
  fetched 34000 records so far...
  fetched 36000 records so far...
  fetched 38000 records so far...
  fetched 40000 records so far...
  fetched 42000 records so far...
  fetched 44000 records so far...
  fetched 46000 records so far...
  fetched 48000 records so far...
  fetched 50000 records so far...
  fetched 52000 records so far...
  fetched 54000 records so far...
  fetched 56000 records so far...
  fetched 58000 records so far...
  fetched 60000 re

In [ ]:
# Parsing into a GeoDataFrame
records = []
for f in all_features:
    attr = f["attributes"]
    geom = f.get("geometry", {})
    attr["longitude"] = geom.get("x")
    attr["latitude"]  = geom.get("y")
    records.append(attr)

df = pd.DataFrame(records)
print("Columns:", df.columns.tolist())
print("Well types:", df["type"].value_counts().head(10))

# Convert to GeoDataFrame — drop rows missing coordinates
df = df.dropna(subset=["longitude", "latitude"])
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs="EPSG:4326"
)

print(f"GeoDataFrame shape: {gdf.shape}")
gdf.head(3)

Columns: ['OBJECTID', 'id', 'name', 'type', 'status', 'sub_type_code', 'ogrid', 'ogrid_name', 'district_code', 'district', 'county_code', 'county', 'ulstr', 'latitude', 'longitude', 'projection', 'directional_status', 'details', 'files', 'year_spudded', 'lease_type', 'measured_vertical_depth', 'true_vertical_depth', 'pool_id_list', 'last_production_date', 'plug_date', 'GlobalID']
Well types: type
Oil                    82942
Gas                    47919
Injection               6104
Salt Water Disposal     1795
CO2                      999
Miscellaneous            386
Water                    144
C02                       34
Name: count, dtype: int64
GeoDataFrame shape: (140323, 28)


,OBJECTID,id,name,type,status,sub_type_code,ogrid,ogrid_name,district_code,district,...,files,year_spudded,lease_type,measured_vertical_depth,true_vertical_depth,pool_id_list,last_production_date,plug_date,GlobalID,geometry
0,1,30-045-08708,DUSTIN #001,Gas,Active,,372171,HILCORP ENERGY COMPANY,3,Aztec,...,http://ocdimage.emnrd.nm.gov/imaging/WellFileV...,1961,Private,6130,6130,[71599] BASIN DAKOTA (PRORATED GAS),1769929200000,253402239600000,{164D4436-E73B-4404-B5A2-877F27CAEE22},POINT (-108.14236 36.75282)
1,2,30-005-61101,SLAYTON FEDERAL #001,Gas,Plugged (site released),,330238,"Solis Partners, L.L.C.",2,Artesia,...,http://ocdimage.emnrd.nm.gov/imaging/WellFileV...,9999,Federal,5000,5000,"[82730] PECOS SLOPE, ABO (GAS)",988700400000,253402239600000,{DD846FAC-9CAE-4B50-8749-90B417BFA5DD},POINT (-104.25656 33.65439)
2,3,30-005-62477,IRWIN FEDERAL #001,Gas,Active,,330238,"Solis Partners, L.L.C.",2,Artesia,...,http://ocdimage.emnrd.nm.gov/imaging/WellFileV...,1987,Federal,0,99999,"[82730] PECOS SLOPE, ABO (GAS)",1767250800000,253402239600000,{DBD9BD69-024A-4FE9-85FD-E71395E355B6},POINT (-104.25639 33.70194)


# Export the wells data

In [20]:
df.to_csv('OCD_wells_data.csv', index=False)

In [22]:
from google.colab import files
files.download('OCD_wells_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Salt water disposal

In [ ]:
# 3. Split into Oil wells and SWD wells
# Check the exact string for water wells first:
print(df["type"].unique())   # adjust filter below to match

oil_wells = gdf[gdf["type"].str.upper().str.contains("OIL",  na=False)]
swd_wells = gdf[gdf["type"].str.upper().str.contains("WATER", na=False)]

print(f"Oil wells: {len(oil_wells):,}")
print(f"SWD wells: {len(swd_wells):,}")

['Gas' 'Oil' 'Injection' 'CO2' 'Salt Water Disposal' 'Miscellaneous'
 'Water' 'C02']
Oil wells: 82,942
SWD wells: 1,939


In [ ]:
m = folium.Map(location=[34.5, -106.0], zoom_start=7, tiles="CartoDB positron")

# SWD wells — teal markers
for _, row in swd_wells.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3, color="#1D9E75", fill=True, fill_opacity=0.7,
        popup=f"SWD: {row.get('WellName','')}"
    ).add_to(m)

# Oil wells — small grey dots (many thousands, so use smaller radius)
for _, row in oil_wells.sample(min(3000, len(oil_wells))).iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=1, color="#888780", fill=True, fill_opacity=0.4
    ).add_to(m)

folium.LayerControl().add_to(m)
m.save("ocd_wells_map.html")
m   # renders inline in Colab


In [ ]:
oil_wells.columns

Index(['OBJECTID', 'id', 'name', 'type', 'status', 'sub_type_code', 'ogrid',
       'ogrid_name', 'district_code', 'district', 'county_code', 'county',
       'ulstr', 'latitude', 'longitude', 'projection', 'directional_status',
       'details', 'files', 'year_spudded', 'lease_type',
       'measured_vertical_depth', 'true_vertical_depth', 'pool_id_list',
       'last_production_date', 'plug_date', 'GlobalID', 'geometry'],
      dtype='object')